# Defining global variables

In [1]:
REPO_NAME = 'Textual_Analysis_in_Finance'
BASE_DIR = f'/kaggle/working/{REPO_NAME}'
WEEK = 6

In [2]:
import warnings
warnings.filterwarnings('ignore')

# Clone the lecture's git repo

In [3]:
!git clone https://github.com/minhtriphan/{REPO_NAME}.git
%cd {REPO_NAME}

Cloning into 'Textual_Analysis_in_Finance'...
remote: Enumerating objects: 306, done.
remote: Counting objects: 100% (306/306), done.
remote: Compressing objects: 100% (229/229), done.
remote: Total 306 (delta 108), reused 198 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (306/306), 9.93 MiB | 20.50 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/kaggle/working/Textual_Analysis_in_Finance


# Retrieval-Augmented Generation

In this notebook, we will build our own RAG system from scratch. First, to do that, we need a package call `faiss-cpu` to do the semantic search. This package can handle the searching of chunks in the document that are most similar to the question

In [4]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 83.2 MB/s eta 0:00:00


## Load the document we want to analyze

We will use RAG to analyse the Tesla's 10K form in 2025, load it here

In [5]:
tesla_10k_2025 = f'{BASE_DIR}/Data/FinancialStatements/20250130_10-K_edgar_data_1318605_0001628280-25-003063.txt'
with open(tesla_10k_2025, 'r') as f:
    content = f.read()

## Design the text splitter

Write the text splitter function, then use it to split the document into chunks

In [6]:
from typing import List

def text_splitter(text: str, chunk_size: int = 1000, chunk_overlap: int = 200, length_func: object = len) -> List:
    chunks = []

    start = 0
    while start < length_func(text):
        # Get chunk
        chunk = text[start : (start + chunk_size)]
        
        # Store the chunk
        chunks.append(chunk)
        
        # Find a new starting position
        start = start + chunk_size - chunk_overlap

    return chunks

chunks = text_splitter(content)

## Initialize the embedding model, then encode the chunks

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

embeddings = model.encode(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Generate an embedding database using FAISS (Facebook AI Similarity Search)

This step will also search for chunks that are similar to the prompt

In [8]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

#### Let's say, we have the below example

In [9]:
# An example question
question = 'What are Tesla\'s future plan in 2026 regarding capital expenditure?'
query_embedding = model.encode([question])

# Find the top k most similar chunks to the question
distances, indices = index.search(
    query_embedding.astype('float32'),
    k = 3
)

retrieved_chunks = [chunks[i] for i in indices[0]]

for chunk in retrieved_chunks:
    print('*' * 20)
    print(chunk)

********************
telligence enabled training and products, and the pace of our capital spend may vary depending on overall priority among projects, the pace at which we meet milestones, production adjustments to and among our various products, increased capital efficiencies and the addition of new projects. Owing and subject to the foregoing as well as the pipeline of announced projects under development, all other continuing infrastructure growth and varying levels of inflation, we currently expect our capital expenditures to exceed 11.00 billion in 2025 and in each of the following two fiscal years. 
 Our business has generally been consistently generating cash flow from operations in excess of our level of capital spend, and with better working capital management resulting in shorter days sales outstanding than days payable outstanding, our sales growth is also generally facilitating positive cash generation. We have and will continue to utilize such cash flows, among other thin

## Pick the LLM to generate the response

Choose an LLM, write a function to generate responses from the prompt

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BACKBONE = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
model = AutoModelForCausalLM.from_pretrained(BACKBONE, pad_token_id = tokenizer.eos_token_id)

def generate_text(prompt, model, tokenizer, temperature, max_new_tokens = 500, device = 'cpu'):
    # Tokenize the prompt
    encoded_item = tokenizer(
        prompt,
        return_attention_mask = True,
        return_tensors = 'pt'
    )

    # Move the input and model to the device
    model.to(device)
    input_ids = encoded_item['input_ids'].to(device)
    attention_mask = encoded_item['attention_mask'].to(device)

    # Generate the new text
    with torch.no_grad():
        generated_text = model.generate(
            input_ids = input_ids,
            attention_mask = attention_mask,
            temperature = temperature,
            max_new_tokens = max_new_tokens,
            pad_token_id = tokenizer.eos_token_id
        )[0]

    # Discard the prompt from the generated text
    generated_text = generated_text[attention_mask.sum(dim = 1)[0]:]

    # Decode
    return tokenizer.decode(generated_text, skip_special_tokens = True)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

#### Construct the prompt and get the response

In [11]:
context = '\n\n'.join(retrieved_chunks)

prompt = f'''
You are a senior investment analyst at a hedge fund, who is considering to invest in Tesla. 

Your task is to assess Tesla's investment plan. Answer the following question given the context.

Context:
{context}

Question:
{question}

Answer:
'''
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
response = generate_text(prompt, model, tokenizer, 0.7, max_new_tokens = 500, device = device)

print('Response:\n')
print(response)

Response:

Tesla expects its capital expenditures to exceed $11 billion in 2025 and in each of the following two fiscal years. The company aims to maintain a consistent level of cash flow from operations exceeding its capital spend, thanks to better working capital management and sales growth. Additionally, they intend to invest in expanding operations, reducing costs, and increasing delivery capabilities through R&D efforts aimed at accelerating AI, software, and fleet-based profits. Their current strategy includes leveraging existing cash reserves and enhancing human capital resources to support these goals. However, specific details about how exactly this plan will be executed remain undisclosed. They also mention ongoing discussions with the board regarding ESG impacts, initiatives, and priorities, indicating that sustainability considerations play a significant role in their long-term planning. Nevertheless, without more detailed information, it's challenging to predict exact futu